# Bronze Layer — Cryptocurrency Market Data

## Purpose
The Bronze layer ingests cryptocurrency market observations from JSON
source files and stores them in a Delta table.

The pipeline uses an explicit schema, validates source records,
quarantines invalid data, and supports repeatable loading.

## Target Table
`workspace.default.bronze_coin_market_raw`

## Business Key
`coin_id` + `observation_timestamp`

## Data Dictionary

| Column | Data Type | Description | Key |
|---|---|---|---|
| coin_id | STRING | Cryptocurrency identifier | Composite key |
| source_file | STRING | Source JSON file path | No |
| timestamp_ms | BIGINT | Original observation timestamp in milliseconds | No |
| price_usd | DOUBLE | Cryptocurrency price in USD | No |
| market_cap_usd | DOUBLE | Market capitalization in USD | No |
| total_volume_usd | DOUBLE | Trading volume in USD | No |
| load_timestamp | TIMESTAMP | Time the record was loaded into Bronze | No |
| observation_timestamp | TIMESTAMP | Converted market observation timestamp | Composite key |

## Data Quality Rules
- Validate incoming JSON files against the expected structure.
- Skip metadata files that do not contain market observations.
- Check that price, market capitalization, and volume arrays are aligned.
- Quarantine malformed files and invalid observations.
- Check duplicate business keys before loading.
- Detect schema drift: a value that is no longer a number sends the whole file to `bronze_file_quarantine`; a new unexpected field is loaded safely and reported in the log's `error_message`.

## Quarantine tables
- `bronze_file_quarantine`: whole files that are malformed or drifted.
- `bronze_quarantine`: single observations with null / NaN / negative values.

## Idempotency
MERGE updates a matched row only when a value changed, so re-running the same files gives 0 inserted and 0 updated.

## Loading Strategy
The pipeline supports full and incremental loads.

Delta MERGE uses `coin_id` and `observation_timestamp` to update
existing observations and insert new ones.

Execution results are recorded in
`workspace.default.pipeline_execution_logs`.

In [0]:
import os
import uuid
from datetime import datetime, timezone

# Pipeline parameters
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental"])
dbutils.widgets.text("processing_date", "")
dbutils.widgets.text("source_folder", "")

load_type = dbutils.widgets.get("load_type")
processing_date = dbutils.widgets.get("processing_date").strip()
source_folder = dbutils.widgets.get("source_folder").strip()

# An incremental load with no date means "today" (UTC)
if load_type == "incremental" and not processing_date:
    processing_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    print("No processing_date given; using today (UTC):", processing_date)

if processing_date:
    datetime.strptime(processing_date, "%Y-%m-%d")   # fails on a bad format

# Find the project's data folder from this notebook's own location,
# so it works no matter which folder Databricks runs the notebook from.
try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    repo_root = "/Workspace" + "/".join(notebook_path.split("/")[:-3])
except Exception:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
data_root = repo_root + "/data/samples"

# Determine input folder (source_folder wins, then the load type)
# source_folder may be absolute (/Workspace/...) or relative to data/samples,
# e.g. "incremental_load_250/2026-10-10" or "drift_test"
if source_folder:
    input_folder = source_folder if source_folder.startswith("/") else os.path.join(data_root, source_folder)
elif load_type == "full":
    input_folder = data_root + "/full_load_250"
else:
    input_folder = data_root + "/incremental_load_250/" + processing_date

# Delta table names
bronze_table_name = "workspace.default.bronze_coin_market_raw"
quarantine_table_name = "workspace.default.bronze_quarantine"
file_quarantine_table_name = "workspace.default.bronze_file_quarantine"
log_table_name = "workspace.default.pipeline_execution_logs"

# Execution tracking
pipeline_run_id = str(uuid.uuid4())
pipeline_start_time = datetime.now(timezone.utc)

print("Load type:", load_type)
print("Processing date:", processing_date or "n/a")
print("Data root:", data_root)
print("Input folder:", input_folder)
print("Folder exists:", os.path.isdir(input_folder))
print("Pipeline run ID:", pipeline_run_id)

In [0]:
import os
import json
from pyspark.sql.types import StructType, StructField, ArrayType, DoubleType

# Explicit JSON schema (schema-on-read, no inference)
observation_schema = ArrayType(ArrayType(DoubleType()))

coin_schema = StructType([
    StructField("prices", observation_schema, True),
    StructField("market_caps", observation_schema, True),
    StructField("total_volumes", observation_schema, True)
])

EXPECTED_FIELDS = ["prices", "market_caps", "total_volumes"]

# Expected metadata files
metadata_filenames = {"download_report.json", "coin_manifest.json"}

valid_file_paths = []
metadata_file_paths = []
file_rejection_reasons = {}
drift_warnings = []


def is_number_or_null(value):
    return value is None or (isinstance(value, (int, float)) and not isinstance(value, bool))


# Check that the source folder exists
if not os.path.isdir(input_folder):
    raise FileNotFoundError("Folder not found: " + input_folder)

# Discover and validate JSON files
for folder_path, subfolders, filenames in os.walk(input_folder):
    for filename in filenames:
        if not filename.endswith(".json"):
            continue

        file_path = os.path.join(folder_path, filename)

        if filename in metadata_filenames:
            metadata_file_paths.append(file_path)
            continue

        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            if not isinstance(data, dict):
                raise ValueError("JSON root is not an object")

            if not all(isinstance(data.get(x), list) for x in EXPECTED_FIELDS):
                raise ValueError("Missing or invalid observation arrays")

            # Type drift: every item must be a [timestamp, value] pair of numbers
            for field in EXPECTED_FIELDS:
                for pair in data[field]:
                    if not (isinstance(pair, list) and len(pair) == 2
                            and all(is_number_or_null(v) for v in pair)):
                        raise ValueError(
                            "Schema drift: '" + field + "' contains an item that is not a [timestamp, number] pair"
                        )

            # New-field drift: loaded safely (extra fields are ignored) but reported
            extra_fields = sorted(set(data.keys()) - set(EXPECTED_FIELDS))
            if extra_fields:
                drift_warnings.append(filename + ": new fields " + str(extra_fields))

            valid_file_paths.append(file_path)

        except (OSError, ValueError, TypeError) as error:
            file_rejection_reasons[file_path] = [str(error)]

valid_file_paths = sorted(valid_file_paths)
drift_message = ("DRIFT WARNING: " + "; ".join(drift_warnings))[:1000] if drift_warnings else None

print("Valid cryptocurrency files:", len(valid_file_paths))
print("Metadata files skipped:", len(metadata_file_paths))
print("Files requiring quarantine:", len(file_rejection_reasons))
print("Files with new (unexpected) fields:", len(drift_warnings))

In [0]:
from pyspark.sql import functions as F

# Convert relative file paths into absolute paths for Spark
spark_file_paths = [
    "file:" + os.path.abspath(path)
    for path in valid_file_paths
]

# Stop if no valid cryptocurrency files were found
if not spark_file_paths:
    raise ValueError("No valid cryptocurrency JSON files found.")

# Read JSON using the explicit schema
raw_coins_df = (
    spark.read
    .schema(coin_schema)
    .option("multiLine", "true")
    .json(spark_file_paths)
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn(
        "coin_id",
        F.regexp_extract(
            F.col("source_file"),
            r"([^/]+)\.json$",
            1
        )
    )
)

print("JSON documents loaded:", raw_coins_df.count())

print(
    "Unique cryptocurrencies:",
    raw_coins_df.select("coin_id").distinct().count()
)

display(raw_coins_df.select("coin_id", "source_file").limit(10))


In [0]:
from pyspark.sql import functions as F

# Count observations in each array
length_check_df = (
    raw_coins_df
    .withColumn("price_count", F.size("prices"))
    .withColumn("market_cap_count", F.size("market_caps"))
    .withColumn("volume_count", F.size("total_volumes"))
)

# Check array lengths
length_mismatch_df = length_check_df.filter(
    (F.col("price_count") != F.col("market_cap_count")) |
    (F.col("price_count") != F.col("volume_count"))
)

length_problems = length_mismatch_df.count()

# Compare timestamps at matching array positions
alignment_df = (
    length_check_df
    .select(
        "coin_id",
        "source_file",
        "price_count",
        "market_cap_count",
        "volume_count",
        F.posexplode_outer("prices").alias(
            "position", "price_pair"
        ),
        "market_caps",
        "total_volumes"
    )
    .withColumn(
        "timestamps_match",
        F.coalesce(
            (
                F.col("price_pair")[0] ==
                F.col("market_caps")[F.col("position")][0]
            ) &
            (
                F.col("price_pair")[0] ==
                F.col("total_volumes")[F.col("position")][0]
            ),
            F.lit(False)
        )
    )
)

timestamp_problems = alignment_df.filter(
    ~F.col("timestamps_match")
).count()

print("Files with mismatched array lengths:", length_problems)
print("Observations with mismatched timestamps:", timestamp_problems)

In [0]:
from pyspark.sql import functions as F
import os

# Start with malformed files identified in Step 66
rejected_files = {}

for path, reasons in file_rejection_reasons.items():
    full_path = "file:" + os.path.abspath(path)
    rejected_files[full_path] = list(reasons)

# Find files with mismatched array lengths
length_mismatch_paths = [
    row.source_file
    for row in length_mismatch_df
        .select("source_file")
        .distinct()
        .collect()
]

for path in length_mismatch_paths:
    rejected_files.setdefault(path, []).append(
        "Array lengths do not match"
    )

# Find files with mismatched timestamps
timestamp_mismatch_paths = [
    row.source_file
    for row in alignment_df
        .filter(~F.col("timestamps_match"))
        .select("source_file")
        .distinct()
        .collect()
]

for path in timestamp_mismatch_paths:
    rejected_files.setdefault(path, []).append(
        "Observation timestamps do not match"
    )

print("Total files requiring quarantine:", len(rejected_files))

if not rejected_files:
    print("All files passed file-level validation!")
else:
    for path, reasons in rejected_files.items():
        print(path, ":", "; ".join(reasons))

In [0]:
from datetime import datetime, timezone
from delta.tables import DeltaTable
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType
)

# Create the quarantine table if needed
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.bronze_file_quarantine (
    source_file STRING,
    processing_date STRING,
    load_type STRING,
    rejection_reason STRING,
    quarantine_timestamp TIMESTAMP
)
USING DELTA
""")

# Explicit schema for quarantine records
file_quarantine_schema = StructType([
    StructField("source_file", StringType(), False),
    StructField("processing_date", StringType(), True),
    StructField("load_type", StringType(), True),
    StructField("rejection_reason", StringType(), True),
    StructField("quarantine_timestamp", TimestampType(), True)
])

# Prepare records for rejected files
quarantine_records = []

for path, reasons in rejected_files.items():
    quarantine_records.append((
        path,
        processing_date,
        load_type,
        "; ".join(sorted(set(reasons))),
        datetime.now(timezone.utc)
    ))

files_to_quarantine = len(quarantine_records)

# Write only when rejected files exist
if files_to_quarantine > 0:
    file_quarantine_df = spark.createDataFrame(
        quarantine_records,
        schema=file_quarantine_schema
    )

    quarantine_delta = DeltaTable.forName(
        spark,
        file_quarantine_table_name
    )

    merge_condition = (
        "target.source_file = source.source_file "
        "AND target.processing_date = source.processing_date "
        "AND target.load_type = source.load_type"
    )

    (
        quarantine_delta.alias("target")
        .merge(
            file_quarantine_df.alias("source"),
            merge_condition
        )
        .whenMatchedUpdate(set={
            "rejection_reason": "source.rejection_reason",
            "quarantine_timestamp": "source.quarantine_timestamp"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

print("Files requiring quarantine:", files_to_quarantine)
print("Bronze file quarantine processing completed!")

In [0]:
from pyspark.sql import functions as F

# Keep only observations that passed timestamp validation
clean_alignment_df = alignment_df.filter(
    F.col("timestamps_match")
)

# Exclude entire files that failed file-level validation
if rejected_files:
    clean_alignment_df = clean_alignment_df.filter(
        ~F.col("source_file").isin(list(rejected_files.keys()))
    )

# Transform aligned arrays into Bronze observations
bronze_source_df = (
    clean_alignment_df
    .select(
        F.col("coin_id"),
        F.col("source_file"),
        F.col("price_pair")[0].cast("long").alias("timestamp_ms"),
        F.col("price_pair")[1].cast("double").alias("price_usd"),
        F.col("market_caps")[F.col("position")][1]
            .cast("double").alias("market_cap_usd"),
        F.col("total_volumes")[F.col("position")][1]
            .cast("double").alias("total_volume_usd"),
        F.current_timestamp().alias("load_timestamp")
    )
    .withColumn(
        "observation_timestamp",
        F.expr("timestamp_millis(timestamp_ms)")
    )
)

print("Bronze observations prepared:", bronze_source_df.count())

print(
    "Unique cryptocurrencies:",
    bronze_source_df.select("coin_id").distinct().count()
)

display(bronze_source_df.limit(10))

In [0]:
from pyspark.sql import functions as F

# Identify invalid observations
invalid_condition = (
    F.col("coin_id").isNull()
    | (F.trim(F.col("coin_id")) == "")
    | F.col("timestamp_ms").isNull()
    | F.col("observation_timestamp").isNull()
    | F.col("price_usd").isNull()
    | F.isnan("price_usd")
    | (F.col("price_usd") < 0)
    | F.col("market_cap_usd").isNull()
    | F.isnan("market_cap_usd")
    | (F.col("market_cap_usd") < 0)
    | F.col("total_volume_usd").isNull()
    | F.isnan("total_volume_usd")
    | (F.col("total_volume_usd") < 0)
)

# Separate valid and invalid records
invalid_bronze_df = bronze_source_df.filter(invalid_condition)

valid_bronze_df = bronze_source_df.filter(~invalid_condition)

# Check duplicate business keys
duplicate_keys_df = (
    valid_bronze_df
    .groupBy("coin_id", "observation_timestamp")
    .count()
    .filter(F.col("count") > 1)
)

# Display results
print("Total observations:", bronze_source_df.count())
print("Valid observations:", valid_bronze_df.count())
print("Invalid observations:", invalid_bronze_df.count())
print("Duplicate business keys:", duplicate_keys_df.count())

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Create the observation quarantine table if needed
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.bronze_quarantine (
    coin_id STRING,
    source_file STRING,
    timestamp_ms BIGINT,
    observation_timestamp TIMESTAMP,
    price_usd DOUBLE,
    market_cap_usd DOUBLE,
    total_volume_usd DOUBLE,
    rejection_reason STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

# Explain why each invalid observation was rejected
observation_quarantine_df = (
    invalid_bronze_df
    .withColumn(
        "rejection_reason",
        F.when(F.col("coin_id").isNull() | (F.trim("coin_id") == ""), "Missing coin ID")
        .when(F.col("timestamp_ms").isNull() | F.col("observation_timestamp").isNull(), "Invalid timestamp")
        .when(F.col("price_usd").isNull() | F.isnan("price_usd") | (F.col("price_usd") < 0), "Invalid price")
        .when(F.col("market_cap_usd").isNull() | F.isnan("market_cap_usd") | (F.col("market_cap_usd") < 0), "Invalid market cap")
        .when(F.col("total_volume_usd").isNull() | F.isnan("total_volume_usd") | (F.col("total_volume_usd") < 0), "Invalid trading volume")
        .otherwise("Unknown validation error")
    )
    .select(
        "coin_id", "source_file", "timestamp_ms", "observation_timestamp",
        "price_usd", "market_cap_usd", "total_volume_usd",
        "rejection_reason", "load_timestamp"
    )
    .dropDuplicates(["coin_id", "timestamp_ms", "source_file"])
)

invalid_count = observation_quarantine_df.count()

if invalid_count > 0:
    # MERGE (not append) so re-running the same files does not duplicate quarantine rows
    (
        DeltaTable.forName(spark, quarantine_table_name).alias("target")
        .merge(
            observation_quarantine_df.alias("source"),
            """
            target.coin_id <=> source.coin_id
            AND target.timestamp_ms <=> source.timestamp_ms
            AND target.source_file <=> source.source_file
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

print("Invalid observations found:", invalid_count)
print("Observation quarantine processing completed!")

In [0]:
# Stop the MERGE if the source contains duplicate business keys
duplicate_count = duplicate_keys_df.count()

if duplicate_count > 0:
    raise ValueError(f"MERGE stopped: {duplicate_count} duplicate business keys found.")

print("Duplicate business keys:", duplicate_count)
print("Bronze MERGE preparation completed!")

In [0]:
from delta.tables import DeltaTable

bronze_delta = DeltaTable.forName(spark, bronze_table_name)
version_before = bronze_delta.history(1).select("version").first()[0]

measures = ["timestamp_ms", "price_usd", "market_cap_usd", "total_volume_usd"]
changed_condition = " OR ".join(f"NOT (target.{c} <=> source.{c})" for c in measures)

# Insert new observations; update an existing one ONLY if a value changed
(
    bronze_delta.alias("target")
    .merge(
        valid_bronze_df.alias("source"),
        """
        target.coin_id = source.coin_id
        AND target.observation_timestamp = source.observation_timestamp
        """
    )
    .whenMatchedUpdate(
        condition=changed_condition,
        set={
            "source_file": "source.source_file",
            "timestamp_ms": "source.timestamp_ms",
            "price_usd": "source.price_usd",
            "market_cap_usd": "source.market_cap_usd",
            "total_volume_usd": "source.total_volume_usd",
            "load_timestamp": "source.load_timestamp"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

# Real numbers from the Delta transaction history
latest = bronze_delta.history(1).first()
if latest["version"] > version_before and latest["operation"] == "MERGE":
    metrics = latest["operationMetrics"]
    rows_to_insert = int(metrics.get("numTargetRowsInserted", 0))
    rows_to_update = int(metrics.get("numTargetRowsUpdated", 0))
else:
    rows_to_insert, rows_to_update = 0, 0   # MERGE changed nothing

bronze_total = spark.table(bronze_table_name).count()

print("Final Bronze MERGE completed successfully!")
print("Rows inserted:", rows_to_insert)
print("Rows updated:", rows_to_update)
print("Total Bronze records:", bronze_total)

In [0]:
from datetime import datetime, timezone
from delta.tables import DeltaTable
from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, LongType
)

# Create the execution log table if needed
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.pipeline_execution_logs (
    run_id STRING,
    layer STRING,
    table_name STRING,
    load_type STRING,
    processing_date STRING,
    source_folder STRING,
    start_timestamp TIMESTAMP,
    end_timestamp TIMESTAMP,
    status STRING,
    rows_read BIGINT,
    rows_inserted BIGINT,
    rows_updated BIGINT,
    rows_quarantined BIGINT,
    error_message STRING
)
USING DELTA
""")

# Explicit schema for the execution log
log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("layer", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("load_type", StringType(), True),
    StructField("processing_date", StringType(), True),
    StructField("source_folder", StringType(), True),
    StructField("start_timestamp", TimestampType(), True),
    StructField("end_timestamp", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("rows_read", LongType(), True),
    StructField("rows_inserted", LongType(), True),
    StructField("rows_updated", LongType(), True),
    StructField("rows_quarantined", LongType(), True),
    StructField("error_message", StringType(), True)
])

pipeline_end_time = datetime.now(timezone.utc)

# Prepare a successful execution record
log_data = [(
    pipeline_run_id,
    "bronze",
    bronze_table_name,
    load_type,
    processing_date,
    input_folder,
    pipeline_start_time,
    pipeline_end_time,
    "SUCCESS",
    int(bronze_source_df.count()),
    int(rows_to_insert),
    int(rows_to_update),
    int(invalid_count + files_to_quarantine),
    drift_message
)]

execution_log_df = spark.createDataFrame(
    log_data,
    schema=log_schema
)

# MERGE prevents duplicate log entries for the same run ID
log_delta = DeltaTable.forName(spark, log_table_name)

(
    log_delta.alias("target")
    .merge(
        execution_log_df.alias("source"),
        "target.run_id = source.run_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Bronze execution log saved successfully!")

display(
    spark.table(log_table_name)
    .filter(F.col("run_id") == pipeline_run_id)
)

import json
dbutils.notebook.exit(json.dumps({
    "run_id": pipeline_run_id, "status": "SUCCESS",
    "rows_inserted": int(rows_to_insert), "rows_updated": int(rows_to_update),
    "rows_quarantined": int(invalid_count + files_to_quarantine)
}))